In [3]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier # Fallback
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# --- 00 - XGBOOST FALLBACK CHECK ---
try:
    from xgboost import XGBClassifier
    has_xgboost = True
except ImportError:
    has_xgboost = False
    print("⚠️ XGBoost not found. Using GradientBoostingClassifier as the 'Boosting' representative.")

# --- 01 - DATA PREPARATION ---
file_path = r"C:\Users\HaadiyaH\Desktop\Data\structured_endometriosis_data.csv"
df_struct = pd.read_csv(file_path)

X = df_struct.drop('Diagnosis', axis=1)
y = df_struct['Diagnosis']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- 02 - THE BENCHMARKING LOOP ---
models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced'),
    "Random Forest": RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
}

# Add either XGBoost or the Scikit-Learn equivalent
if has_xgboost:
    models["XGBoost"] = XGBClassifier(scale_pos_weight=5, eval_metric='logloss', random_state=42)
else:
    models["Gradient Boosting"] = GradientBoostingClassifier(n_estimators=100, random_state=42)

benchmark_results = []

print("🚀 Starting Symptom Model Benchmarking...")

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    probs = model.predict_proba(X_test_scaled)[:, 1]
    
    benchmark_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "Endo F1-Score": f1_score(y_test, preds),
        "AUC-ROC": roc_auc_score(y_test, probs),
        "Model_Obj": model # Store the object to save later
    })
    print(f"✅ {name} Completed.")

# --- 03 - IDENTIFY WINNER & SAVE ---
results_df = pd.DataFrame(benchmark_results).sort_values(by="Endo F1-Score", ascending=False)
best_model_name = results_df.iloc[0]['Model']
winner_model = results_df.iloc[0]['Model_Obj']

# Save as a single "Expert Bundle"
symptom_bundle = {
    'model': winner_model,
    'scaler': scaler,
    'feature_names': list(X.columns)
}

joblib.dump(symptom_bundle, "LogisticRegression.joblib")

print(f"\n🌟 Winner: {best_model_name}")
print("📂 Saved 'LogisticRegression.joblib' for your Fusion Model.")

⚠️ XGBoost not found. Using GradientBoostingClassifier as the 'Boosting' representative.
🚀 Starting Symptom Model Benchmarking...
✅ Logistic Regression Completed.
✅ Random Forest Completed.
✅ Gradient Boosting Completed.

🌟 Winner: Logistic Regression
📂 Saved 'LogisticRegression.joblib' for your Fusion Model.
